# Session 7 — Evaluating Multi-Agent Systems

**Halvard Works maintenance assistant.** Planner → Diagnostics → Documentation → Maintenance.

Sessions 1–6 evaluated one Deep Research agent. From here on there are four agents, and
the things that go wrong are no longer inside any one of them.

> **The sentence this session adds:** *Every agent you add is a decision you added, and
> every decision is somewhere to be wrong.*

Halvard Works is fictional — the plant, its machines, sensors, manuals, parts and history
were invented for this course. Inspired by publicly described industrial-copilot products;
not affiliated with or endorsed by any vendor.

## Every term, before it is used

| term | in this session it means |
|---|---|
| **agent** | one LLM with its own prompt and its own tools. We have four. |
| **delegation** | the planner deciding which agent handles which subtask. |
| **handoff** | the single string one agent passes to the next. Nothing else crosses. |
| **handoff fact** | a piece of text you assert MUST be in that string. |
| **a run** | one request sent to one arm, once. |
| **arm** | `pipeline` (four agents) or `single` (one agent, five tools). `paired.py` calls this the *version*. |
| **tokens_billed** | output tokens + *uncached* input tokens, summed over the whole trace. Cached reads excluded. Course definition since Session 3. |
| **an evaluator fires** | it returned 0. Confusingly, that is the evaluator working. |

In [ ]:
# [PULL]
# Session 4 onwards this is a repo, not Colab.
!git pull --ff-only
!python check_env.py

In [ ]:
# [PINS]
# Pin verifier. A resolution FAILURE is loud; a resolution SUCCESS that quietly
# installed a different version is silent, and would corrupt every number below.
import importlib.metadata as md
PINS = {'langchain-core': '1.6.1', 'langgraph': '1.2.11', 'langsmith': '0.11.1'}
bad = {p: md.version(p) for p, want in PINS.items() if md.version(p) != want}
print('pins OK' if not bad else f'WRONG VERSIONS: {bad} -- fix before going on')

In [ ]:
# [SETUP]
# Reload the course modules in DEPENDENCY ORDER before importing anything.
#
# A Jupyter kernel caches modules. Edit a file on disk, re-run the cell, and
# you get the version from the first import -- silently, unless the change
# added a NAME, in which case you get an ImportError that looks like the file
# is wrong when it is the kernel that is stale. Dependency order matters:
# reloading plant_agents7 before plant7 re-runs its `from plant7 import ...`
# against the OLD plant7.
import importlib, sys
for _m in ('plant7', 'plant_tools7', 'delegation_rows7', 'plant_agents7',
           'seeds7', 'coord_eval7', 'bench7', 'my_handoffs7'):
    if _m in sys.modules:
        importlib.reload(sys.modules[_m])

import evalkit, bench7, plant7

# A loud staleness check. If this fires, reloading was not enough:
# Kernel -> Restart Kernel, then Run All Above.
assert hasattr(plant7, 'PLANT_ONE_LINER'), (
    'STALE KERNEL: plant7 is an old copy. Kernel -> Restart Kernel, then run from the top.')
print('evalkit', evalkit.__version__, '| bench7', bench7.__version__)

# Conventions #5: LANGSMITH_PROJECT must be set BEFORE anything reads it, or
# traces land in `default` with no error at all.
print('resolved project ->', evalkit.env_setup(bench7.PROJECT))   # must NOT say 'default'

## 1 · The plant

**Halvard Works is a soft-drink bottling plant.** Empty bottles are conveyed to the
filler, filled and capped, blown dry by an air knife, labelled and packed into cases.

It runs **in series**, and that is the only fact you need to judge any recommendation
this session produces: nothing downstream of the filler runs when the filler stops.

Nobody in this room can verify a bearing diagnosis, so the plant is simulated and the
tools are deterministic. But the physics is real, and — this is the part that matters —
**you can check every diagnosis yourself**, because the tools print both halves of the chain.

In [ ]:
# [PLANT]
import plant7

# This prints SLIDE 5. Same headings, same order, same words — because a
# notebook that says the same thing differently is a second source of
# truth wearing a disguise. Both read plant7.py.
print(f'{plant7.PLANT_NAME} — Line 3')
print(plant7.PLANT_ONE_LINER[0].upper() + plant7.PLANT_ONE_LINER[1:])
print()

st = list(plant7.STAGES.items())
W = 26
print('Line 3, in series:')
print('   ' + '    '.join(f'{k.upper():<{W}}' for k, _ in st))
print('   ' + '    '.join(f"{v['does']:<{W}}" for _, v in st))
print('   ' + '    '.join(f"{v['machine']:<{W}}" for _, v in st))
print()
print('Plant-wide:  ' + '   ·   '.join(
    f"{k} ({v['machine']})" for k, v in plant7.UTILITIES.items()))
print()

# The table under it, sorted the same way the slide sorts it: high first.
rows = sorted(plant7.EQUIPMENT.items(),
              key=lambda kv: not kv[1]['criticality'].startswith('high'))
print(f"{'MACHINE':<11}{'WHAT IT IS':<40}WHAT STOPS IF IT DOES")
for mid, eq in rows:
    stops = eq['criticality'].split('—')[-1].strip()
    print(f'{mid:<11}' + f"{eq['name']:<40}" + stops)
print()
print(plant7.DISCLAIMER)

In [ ]:
# [CHECK]
# Check the diagnosis by hand. Two numbers, and this is SLIDE 6.
import json, plant_tools7
kb = json.loads(plant_tools7.equipment_kb.invoke({'machine_id': 'CONVEYOR'}))
sn = json.loads(plant_tools7.sensor_history.invoke({'machine_id': 'CONVEYOR'}))

print('THE SPEC SHEET SAYS')
print(f"  if this bearing were worn it would thump {kb['bearing_thumps_per_turn']} times per turn")
print()
print('THE SENSOR SAYS')
print(f"  it is thumping {sn['measured_thumps_per_turn']} times per turn")
print(f"  and running at {sn['bearing_temp_c']['day_14']} C, alarm {kb['alarm_temp_c']} C, "
      f"up from {sn['bearing_temp_c']['day_1']} C a fortnight ago")
print()
print('SO       the bearing is worn.   Fault code: BEARING-WEAR')

## 2 · What the assistant is for

Nothing above has said what this thing *does*. Here it is.

A maintenance engineer at Halvard Works types **one sentence**:

> *"The filler infeed conveyor is running hot and the vibration alarm keeps tripping.
> What is wrong and what do we do about it?"*

and the assistant answers with four things: **what is wrong** (`BEARING-WEAR`), **which
part** (`SKF-6208`), **what to do** (`replace`), and **the manual section that says so**.

Without it, that engineer walks to the condition-monitoring terminal, then to the manual
shelf, then to the stores counter, and joins it up in their head.

Every row in the benchmark has that shape. And every evaluator you write today asks a
question about **how that answer was reached** — not about whether it reads well.

The only real question in this session: that answer can be produced by **four agents or
by one**. Which should you ship?

## 3 · Four agents — and an open question, not a verdict

Two serious answers to the same question are shipping today.

**One agent.** LangChain's current multi-agent guidance: *"Use single agent with
middleware for most handoffs use cases — it's simpler."* The `langgraph-supervisor`
package now points to the supervisor-via-tools pattern instead.

**Several agents.** Claude Code ships subagents — each with its own context window, its
own tools, chosen by the parent model from a description.

Both are current. Both are documented. Neither is a mistake.

**So why build one?** Because *the skill is the evaluation, not the architecture* — you
cannot evaluate a coordination failure you never had, and the moment a system has more
than one moving part, these are the checks it needs. "Most use cases" is a default, not
a prohibition; yours may be one of the others, and only a measurement on your own data
says which. And you will inherit multi-agent systems you did not choose.

At the end of this notebook you will have an interval measured on this plant. It will not
settle the industry. It settles Halvard Works.

In [ ]:
# [AGENTS]
import plant_agents7 as pa

# The planner's RULES, in full. (Slicing the prompt by character count
# printed a sentence starting mid-word; extract the section instead.)
rules = pa.PLANNER_PROMPT.split('RULES')[1].split('Reply with JSON')[0]
print('RULES the planner is given:')
print(rules.rstrip())

print('Who may touch what:')
for agent, tools in pa.TOOLSETS.items():
    print(f"  {agent:14s} {', '.join(t.name for t in tools)}")
print(f"  {'planner':14s} no tools — it only decides")

In [ ]:
# [LIVE]
# The healthy pipeline, one request. This costs money -- it is four agents.
row = __import__('delegation_rows7').BY_ID['HW-001']
print('REQUEST:', row['request'], '\n')
out = pa.run_pipeline(row['request'], impl='llm')
print('PATH :', ' -> '.join(out['agent_calls']))
print('TAIL :', out['tail'])
print()
for h in out['handoffs']:
    print(f"  {h['from']:14s} -> {h['to']:14s}  {len(h['payload'])} chars handed over")

## 4 · Finding four agents in one trace

Every specialist is a `create_agent` graph, so **every one of them emits spans with the
same default names**. Matching on span names would attribute one agent's tool calls to
another and never raise — the same shape as the bug in conventions #17.

So each invocation is *tagged*: `metadata.agent_name`. Read the tag, never the name.

In [ ]:
# [TRACE]
import coord_eval7, warnings
from langsmith import Client
evalkit.flush_traces()          # conventions #6: traces send on a background thread
client = Client()
root_id = evalkit.find_trace_root(client, since=None, project=bench7.PROJECT)
root, note = evalkit.read_run_when_ready(client, root_id)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')            # conventions #7: get_run_url is deprecated; keep it
    print(client.get_run_url(run=root, project_name=bench7.PROJECT))
agents = coord_eval7.agents_from_run(root)
print(f'spans in this one request : {evalkit.count_spans(root)}')
print(f'of which agents           : {len(agents)}   {agents}')
print(coord_eval7.online_agent_census(root)['comment'])
print(note or '')

**Open that trace.** It is a live run — real seconds, real tokens, real cost.

You can see `agent:diagnostics` and `agent:documentation` because the code sets
`run_name`. Now expand one of them: its children are `model`, `tools`,
`manual_search` — and **none of them carries an agent name.** A run name is not
inherited by children; metadata is. That is why `agents_from_run` reads
`metadata.agent_name`, and why it counts only the **outermost** tagged span.

Three things to find, each with a number:

1. How many spans does *one* request produce, and how many of them are agents? The
   cell above prints both. Expect **dozens of spans and a handful of agents** — the
   exact numbers depend on how the planner routed. **Every coordination question you
   have — who ran, in what order, what got handed over — lives at the handful.** All
   the rest is what those agents did internally, and finding the handful among the
   dozens is exactly what the metadata tag is for.
2. Which agent cost the most? Hover a span — LangSmith breaks down tokens and dollars.
3. What did the planner actually output? A plan, with a subtask per step.

## 5 · Four ways a pipeline breaks that have nothing to do with any agent

| seed | what is injected |
|---|---|
| `wrong_delegation` | diagnostics work is sent to the documentation agent |
| `lost_handoff` | the fault code is dropped on the way to the next agent |
| `redundant_call` | diagnostics runs twice on the same subtask |
| `delegation_loop` | documentation and maintenance ping-pong until the cap |

None of these touches an agent's prompt. They are injected into the graph's **state**, so
the agents are provably not at fault — which is what makes an evaluator that fires here a
measurement of *coordination* and nothing else.

These run on the deterministic stub: no key, no spend, no waiting.

In [ ]:
# [SEEDS]
import seeds7
# Self-contained on purpose: the live cell above is the one you skip when
# the API is down, and this block must still run when it was skipped.
row = __import__('delegation_rows7').BY_ID['HW-001']
for name, meta in seeds7.SEEDS.items():
    r = pa.run_pipeline(row['request'], impl='stub', seed=seeds7.get(name))
    print(f"{name:18s} {' -> '.join(r['agent_calls'])}")
    print(f"{'':18s} {meta['one_line']}")

## 6 · Four evaluators — where to look, and what counts as failing

| evaluator | where it looks | it fails when |
|---|---|---|
| `delegation_accuracy` | the sequence of agent invocations | a forbidden agent ran, an expected one did not, or the order is wrong |
| `handoff_integrity` | **the payload the receiver was handed** | a fact the sender established is absent from what the next agent got |
| `agent_no_redundancy` | how many times each agent ran | more invocations than the plan needed |
| `no_delegation_loop` | the shape of the sequence | a cycle repeats three or more times |

Two design notes worth arguing with:

- Redundancy is counted against the row's `expected_calls`, **not** against "more than
  one". Row HW-006 covers two machines and correctly runs diagnostics twice. A check that
  cannot tell those apart fires on a healthy run — the classic evaluator bug from Session 2.
- `handoff_integrity` asserts against **what the receiver heard**, not against the final
  answer. An answer can be right while the pipeline was broken.

In [ ]:
# [MATRIX]
import coord_eval7
from delegation_rows7 import ROWS
EV = [f.__name__ for f in coord_eval7.OFFLINE]
print(f"{'seed':18s} " + ' '.join(f'{e[:13]:>14s}' for e in EV))
for seed in ('healthy',) + seeds7.BROKEN:
    n = {e: 0 for e in EV}
    for r in ROWS:
        res = coord_eval7.run_all(pa.run_pipeline(r['request'], impl='stub',
                                                  seed=seeds7.get(seed)), r)
        for e in EV: n[e] += res[e]['score']
    print(f"{seed:18s} " + ' '.join(f'{n[e]:>10d}/{len(ROWS):<3d}' for e in EV))

Look at the `lost_handoff` row before going on.

Delegation passes. Redundancy passes. No loop. **And the outcome grader passes too.**
One evaluator noticed. Say what that means about the other four.

## 7 · Hands-on — write a handoff assertion

Open **`my_handoffs7.py`**. It is the only file you edit today.

For two rows, decide what must survive each handoff, and write it down. Then predict —
*before* you run anything — which seeded failure your assertion catches and which it misses.

The trap: pick a fact that would be there anyway and your assertion can never fail.

In [ ]:
# [SCREEN]
!python screen_my_handoffs.py

**Hands up: whose assertion came back `DECORATIVE`?**

A `DECORATIVE` assertion passes the healthy run *and* the broken one. It is not a weak
test. It is not a test. Session 5 established the same thing about benchmark rows; this is
that lesson applied to a boundary instead of an answer.

## 8 · Was four agents worth it?

**What is being compared:** two arms — `pipeline` (four agents) against `single` (one agent
holding all five tools, same capability, same plant). **A run** is one request sent to one
arm once. The instructor ran every row several times against both arms; those runs ship as
`runs7.json`, so this costs you nothing.

**What the interval means.** For each request we average its repetitions within an arm,
then take the difference per request, then average those differences. The bracket is a 95%
interval on that average. *If it crosses zero, the two arms are tied* — and Session 6's
tie rule applies: **the tie goes to the cheaper agent.**

**Read this before the numbers.** That four agents cost more than one is arithmetic, not a
finding — four agents make four times as many model calls. The only interesting column is
whether the pipeline got more rows *right*.

In [ ]:
# [COMPARE]
import json, os, paired
if not os.path.exists('runs7.json'):
    print('runs7.json missing -- `git pull`. Nothing below is real without it.')
else:
    all_runs = json.load(open('runs7.json'))['runs']
    # runs7.json holds THREE measurements: the stub seed matrix, the live
    # tail-contract probe, and the live comparison. Only the last one is a
    # paired pipeline-vs-single experiment. Pooling them averages two
    # different pipeline batches against one single batch -- which is
    # exactly how this cell once printed -25% while the slide said -4%.
    runs = [r for r in all_runs if r.get('phase') == 'comparison']
    print(f'{len(runs)} runs in phase=comparison, out of {len(all_runs)} in the file')
    for metric in ('outcome_match', 'tokens_billed', 'latency_s', 'n_tool_calls'):
        p = paired.paired(runs, 'pipeline', 'single', metric)
        if not p.n_rows:
            print(f'{metric:16s} no paired rows'); continue
        tie = 'TIE (crosses zero)' if p.lo <= 0 <= p.hi else 'SEPARATED'
        band = f'[{p.pct_lo:+.0f}, {p.pct_hi:+.0f}]'
        print(f'{metric:16s} single vs pipeline {p.pct:+6.0f}% {band}  n={p.n_rows}  {tie}')

In [ ]:
# [REPLAY]
# Put the instructor's runs into YOUR workspace. Zero model cost.
# It creates a dataset `s7-halvard-delegation` and SEVEN experiments:
#   s7-replay-pipeline          vs  s7-replay-single    <- live, this is slide 20
#   s7-replay-pipeline-healthy  vs  the four seeded arms <- stub, this is slide 15
# Those two comparisons do NOT mix: the stub runs carry 0 tokens and 0.01 s.
!python replay7.py --dry
# then, when you are ready:   !python replay7.py

Two traps in the UI you are about to open — **both about the REPLAY, neither true of
the live trace you opened earlier**:

1. **A replayed experiment shows 0 s latency and 0 tokens.** Nothing was called — the
   target returns saved outputs. The real numbers are in the *feedback* columns:
   `tokens_billed`, `latency_s`, `n_agent_calls`, `n_tool_calls`.
2. **The Compare view assumes higher is better, per column, and shows no intervals.** A
   slower, costlier arm renders green. Do not read a winner off that screen.

**Three things to go and find, in LangSmith → Datasets → `s7-halvard-delegation` →
Experiments:**

1. Open `s7-replay-pipeline-lost_handoff` and sort by `handoff_integrity`. How many rows
   scored 0 — and what did the *other four* evaluators score on those same rows?
2. Tick `s7-replay-pipeline` and `s7-replay-single` and hit Compare. Which columns are
   green, and does green mean *better*?
3. Find a row where the two arms disagree on `outcome_match`, open both runs, and say
   what is actually different. It is usually the recommendation, not the diagnosis.

Everything on that screen is true and none of it is a conclusion. The interval is in the
cell above, not in the UI — and the UI is the thing that looks authoritative.

## 9 · What we did not do, and who does it next

Every evaluator today is **code**. Nothing judged whether the diagnosis was any *good* —
only whether the right agents ran, in the right order, without losing anything on the way.
That is deliberate: the cheapest grader is the one with no model in it, and **Session 8**
owns the judges.

The four failures here are also not a taxonomy. They overlap — a loop *is* a kind of
redundancy, and the table above shows both firing. The published attempt at a real
taxonomy of multi-agent failures is **MAST** (arXiv 2503.13657); its categories overlap too.

**Homework** is in your inbox: two more handoff assertions, and one delegation row of your
own for a request the pipeline gets wrong.